Knowledge Distillation

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets

In [16]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


Transforming the Dataset

In [17]:
transforms_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms_cifar)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms_cifar)

Train/Test Split

In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

Model Definitions

In [19]:
# Teacher
class DeepNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# Student
class LightNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

Training

In [20]:
def train(model, train_loader, epochs, learning_rate, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

def test(model, test_loader, device):
    model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

Cross Entropy Runs

Teacher Model Training

In [21]:
torch.manual_seed(42)
nn_deep = DeepNN(num_classes=10).to(device)
train(nn_deep, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_deep = test(nn_deep, test_loader, device)

Epoch 1/10, Loss: 1.3319437458082233
Epoch 2/10, Loss: 0.8653377024718868
Epoch 3/10, Loss: 0.6789069046907108
Epoch 4/10, Loss: 0.536766336016033
Epoch 5/10, Loss: 0.41447498632209073
Epoch 6/10, Loss: 0.307332470212751
Epoch 7/10, Loss: 0.22313353745147702
Epoch 8/10, Loss: 0.1744304174066657
Epoch 9/10, Loss: 0.14038864897606929
Epoch 10/10, Loss: 0.11246638915613484
Test Accuracy: 76.01%


In [22]:
torch.manual_seed(42)
nn_light = LightNN(num_classes=10).to(device)
train(nn_light, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_light = test(nn_light, test_loader, device)

Epoch 1/10, Loss: 1.4680933138293684
Epoch 2/10, Loss: 1.1542431045981014
Epoch 3/10, Loss: 1.024217963218689
Epoch 4/10, Loss: 0.9234951780275311
Epoch 5/10, Loss: 0.8485553267666751
Epoch 6/10, Loss: 0.7812353907643682
Epoch 7/10, Loss: 0.7170767440362964
Epoch 8/10, Loss: 0.6613233782293851
Epoch 9/10, Loss: 0.608586555170586
Epoch 10/10, Loss: 0.5588258687035202
Test Accuracy: 70.10%


In [23]:
torch.manual_seed(42)
nn_light2 = LightNN(num_classes=10).to(device)
train(nn_light2, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_light2 = test(nn_light2, test_loader, device)

Epoch 1/10, Loss: 1.4697755294687607
Epoch 2/10, Loss: 1.1604370947384164
Epoch 3/10, Loss: 1.030743298323258
Epoch 4/10, Loss: 0.9287960050660936
Epoch 5/10, Loss: 0.8482169219295083
Epoch 6/10, Loss: 0.7782020267013394
Epoch 7/10, Loss: 0.7104435928947176
Epoch 8/10, Loss: 0.6562724180538636
Epoch 9/10, Loss: 0.6030614095575669
Epoch 10/10, Loss: 0.5524405656415788
Test Accuracy: 70.34%


In [24]:
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy: {test_accuracy_light:.2f}%")
print(f"Student accuracy: {test_accuracy_light2:.2f}%")

Teacher accuracy: 76.01%
Student accuracy: 70.10%
Student accuracy: 70.34%


Knowledge Distillation

In [25]:
def train_knowledge_distillation(teacher, student, train_loader, epochs, learning_rate, T, soft_target_loss_weight, ce_loss_weight, device):
    ce_loss = nn.CrossEntropyLoss()
    kl_loss = nn.KLDivLoss(reduction="batchmean")
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.eval()
    student.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.no_grad():
                teacher_logits = teacher(inputs)

            student_logits = student(inputs)

            soft_targets = nn.functional.softmax(teacher_logits / T, dim=-1)
            soft_prob = nn.functional.log_softmax(student_logits / T, dim=-1)
            soft_targets_loss = kl_loss(soft_prob, soft_targets) * (T ** 2)

            label_loss = ce_loss(student_logits, labels)
            loss = (soft_target_loss_weight * soft_targets_loss) + (ce_loss_weight * label_loss)

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

In [26]:
train_knowledge_distillation(teacher=nn_deep, student=nn_light, train_loader=train_loader, epochs=20, learning_rate=0.001, T=3, soft_target_loss_weight=0.3, ce_loss_weight=0.7, device=device)
test_accuracy_light_ce_and_kd = test(nn_light, test_loader, device)

print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_light2:.2f}%")
print(f"Student accuracy with CE + KD: {test_accuracy_light_ce_and_kd:.2f}%")

Epoch 1/20, Loss: 1.4243388624142503
Epoch 2/20, Loss: 1.2756770347695217
Epoch 3/20, Loss: 1.1777322546905264
Epoch 4/20, Loss: 1.1152485712714817
Epoch 5/20, Loss: 1.039402597852985
Epoch 6/20, Loss: 0.9716293822469004
Epoch 7/20, Loss: 0.9217503015952342
Epoch 8/20, Loss: 0.8759382309206306
Epoch 9/20, Loss: 0.8365134362064665
Epoch 10/20, Loss: 0.8058738357880536
Epoch 11/20, Loss: 0.763834798427494
Epoch 12/20, Loss: 0.7328172603531566
Epoch 13/20, Loss: 0.7037083537072477
Epoch 14/20, Loss: 0.6856960188549803
Epoch 15/20, Loss: 0.6559408604336516
Epoch 16/20, Loss: 0.6456176155363508
Epoch 17/20, Loss: 0.6177832913368254
Epoch 18/20, Loss: 0.6138576106799533
Epoch 19/20, Loss: 0.5940086281360568
Epoch 20/20, Loss: 0.5838085605055475
Test Accuracy: 70.50%
Teacher accuracy: 76.01%
Student accuracy without teacher: 70.34%
Student accuracy with CE + KD: 70.50%
